# Подключение базы к ИИ в другом проекте

Короткая инструкция: как подключить базу знаний к Claude Code (или другому MCP-клиенту) в любом проекте и перенести туда документацию этого проекта. Подробности параметров — `development/mcp-kb/setup#clients` и `development/mcp-kb/api#connection`.

## Перед началом

- Jupyter запущен: `start_jupyter.bat` в папке проекта (на телефоне — `start_jupyter.sh`). MCP работает только пока он запущен.
- Готовые команды `claude mcp add …` с проставленным токеном (все профили `kb` и `kb-admin`) плагин печатает в консоли Jupyter после старта — просто скопировать нужную (`development/mcp-kb/setup#token`).
- Токен отдельно — строка `[kb] token:` в консоли или файл `jupyter_data/config/token.txt` в папке проекта. Он постоянный: один раз прописал в клиенте — и всё.

## Claude Code

В каталоге нового проекта:
```
claude mcp add --transport http -s local kb "http://127.0.0.1:8888/mcp/kb?profile=full&lang=ru&off=structured&token=<токен>"
```
- `-s local` — сервер виден только в этом проекте; `-s user` — во всех проектах сразу.
- `off=structured` для Claude Code обязателен, иначе модель видит JSON вместо плоского текста.
- Проверка: `/mcp` → `kb` connected; затем попросить модель `get("")` — придёт список областей.

**Индексатор (по желанию):** то же для `kb-admin`:
```
claude mcp add --transport http -s local kb-admin "http://127.0.0.1:8888/mcp/kb-admin?lang=ru&off=structured&token=<токен>"
```
и навык `kb-index` — он живёт в базе (`development/mcp-kb/skills/kb-index`). Установить: сказать агенту «установи навык из `development/mcp-kb/skills/kb-index`» — он соберёт `SKILL.md` из статьи по её `#install`.
- Ставится **один раз на машину** в `~/.claude/skills/` — для всех проектов. Уже стоит — в новом проекте нужен только `kb-admin`. Проверка: `kb-index` есть в списке навыков Claude Code.
- После правки статьи-навыка — «обнови навык kb-index».

Агентам, которые только пользуются базой, давать один `kb`.

## Другие клиенты

Любой MCP-клиент с транспортом Streamable HTTP: адрес `http://127.0.0.1:8888/mcp/kb?...&token=<токен>` (или заголовок `Authorization: token <токен>`).
- Слабая или локальная модель — `profile=lite`; средняя — `profile=std`.
- Модель сбивается на английский — `lang=ru`.
- Клиент показывает модели JSON вместо текста — `off=structured`.

## Перенос документации проекта

Как это сделано для документации самого плагина (тема `development/mcp-kb`):
1. Тема проекта: `save_topic("development/<проект>", "<Название>", "для чего проект и что в теме")`.
2. Стартовая статья `development/<проект>/overview`: цели, решения, статус и карта остальных статей. Остальное — по статье на вопрос (хранение, API, запуск и т.п.), секция — один вопрос.
3. В корне проекта — только `README.md` со ссылкой на стартовую статью; старые `.md` с документацией удалить после переноса.
4. В `CLAUDE.md` (или памяти агента) проекта — строка: «Документация проекта — в базе знаний (MCP `kb`), тема `development/<проект>`, начинать с `overview`; правки — только туда».
5. После переноса — `/kb-index`: индексатор расставит теги и проверит шапки.